In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · From a load test to Cloud Run settings

**What you'll learn.** How the overload feedback loop looks in a load test, what admission control does to it,
and how to turn measurements into Cloud Run settings (concurrency, min/max instances, billing mode) and into
the "what breaks first" answer. Code: `scalelab/sim.py`.

> **The one-minute version.** *"I would validate the design with a load test that reproduces the incident mix before
> launch: it gives me the turn duration, which with Little's law gives me in-flight turns, which sizes the
> orchestrator — and it tells me the cap above which shedding is kinder than queueing."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 140)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. Four regimes

The same 120 users against a simulated 3 M TPM pool: naive (retries only), degrade levels + breaker without a cap,
and degrade levels with an in-flight cap of 30. Each run is ~90 virtual seconds (a few wall-clock seconds).

In [ ]:
from scalelab.sim import make_setup, simulate, compare
results = {}
for name, kw, users in [("moderate (20 users)", dict(pool_tpm=3_000_000), 20),
                        ("overload, naive", dict(pool_tpm=3_000_000, naive=True), 120),
                        ("overload, degrade only", dict(pool_tpm=3_000_000, max_inflight=1000), 120),
                        ("overload, degrade + cap 30", dict(pool_tpm=3_000_000, max_inflight=30), 120)]:
    CLOCK.reset(0.02)
    results[name] = await simulate(make_setup(**kw), users=users, duration_s=90, think_s=5)
summary = compare(results).round(3)
summary

In [ ]:
latency_cdfs(results)

In [ ]:
timeline(results["overload, naive"], "naive: the pool saturates, retries pile up, latency explodes")

In [ ]:
timeline(results["overload, degrade + cap 30"], "cap 30: in-flight bounded, level rises, excess shed")

Read the table with the primer's vocabulary: the naive run *is* the feedback loop (429s → retries → long turns →
more in flight); "degrade only" converts it into breaker trips and hard failures; the cap converts it into shed
requests with a `Retry-After` and keeps every admitted turn fast — at a fifth of the cost per turn because most run
on the lite tier.

## 2. From measurements to Cloud Run settings

Take the turn duration from the moderate run, apply Little's law at the planned peak, add headroom, divide by the
concurrency one orchestrator instance can hold (I/O-bound coroutines; ~a few MB of context each).

In [ ]:
from scalelab.capacity import Scenario, plan
s, p = Scenario(), plan(Scenario())
measured = results["moderate (20 users)"].summary()
turn_p50, turn_p95 = measured["p50_s"], measured["p95_s"]
for level in ("average", "peak", "incident"):
    turns_s = p["rates"][level]["turns_per_s"]
    inflight = turns_s * turn_p95                       # size for the tail, not the median
    print(f"{level:9s} {turns_s:5.1f} turns/s × {turn_p95:.1f} s = {inflight:6.0f} in flight → "
          f"{np.ceil(inflight * 1.4 / 80):3.0f} orchestrator instances at concurrency 80 (×1.4 headroom)")

| Setting | Gateway | Orchestrator | Why |
|---|---|---|---|
| Billing | request-based | **instance-based** | checkpoints, compaction and telemetry run after the response |
| Concurrency | 250 (SSE streams are cheap) | 80 (memory per in-flight turn) | Cloud Run max 1,000; 800 req/s per instance |
| Min / max instances | 2 / 100 | 2 / 50 | min for warm capacity; max bounded by regional quota and Direct VPC egress (~100–200/revision) |
| Timeout | 600 s | 600 s (= Pub/Sub ack deadline) | SSE streams are requests; hard ceiling 60 min |
| Autoscaling | 60 % CPU or 60 % of max concurrency | same | scale-out waits max(10 s, 3.5× predicted cold start) |

Cloud Run pricing (Tier 1, Sep 2026): request-based $0.000024/vCPU-s + $0.0000025/GiB-s + $0.40/M requests;
instance-based $0.000018/vCPU-s + $0.000002/GiB-s. The whole fleet is about 0.1 % of the model bill.

## 3. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn — solutions

#### (a) Instances needed

ceil(in-flight × headroom ÷ concurrency), never below `minimum`.

In [ ]:
def instances_needed(inflight, concurrency=80, headroom=1.4, minimum=2):
    return max(minimum, int(np.ceil(inflight * headroom / concurrency)))

In [ ]:
def _a():
    assert instances_needed(125) == 3 and instances_needed(417) == 8 and instances_needed(10) == 2
check("a: instances", _a)

#### (b) Cap from the token budget

How many turns can be in flight before a TPM budget is exhausted: (tpm/60) ÷ (tokens per turn ÷ turn seconds).

In [ ]:
def cap_from_budget(tpm, tokens_per_turn, turn_seconds):
    return (tpm / 60) / (tokens_per_turn / turn_seconds)

In [ ]:
def _b():
    assert abs(cap_from_budget(10_000_000, 2.2 * 5350, 6.0) - p["concurrency"]["max_inflight_for_baseline"]) < 0.01
    assert 20 < cap_from_budget(3_000_000, 2.5 * 3_800, 4.5) < 30
check("b: cap from budget", _b)

#### (c) Cloud Run monthly cost

vCPU-seconds and GiB-seconds for `instances` running all month (730 h) in either billing mode; request-based adds $0.40 per million requests.

In [ ]:
def cloud_run_monthly_usd(vcpu, gib, instances, mode, requests_per_month=0):
    seconds = 730 * 3600 * instances
    if mode == "instance":
        return seconds * (vcpu * 0.000018 + gib * 0.000002)
    return seconds * (vcpu * 0.000024 + gib * 0.0000025) + requests_per_month / 1e6 * 0.40

In [ ]:
def _c():
    assert abs(cloud_run_monthly_usd(2, 2, 3, "instance") - 730*3600*3*(2*0.000018 + 2*0.000002)) < 1e-6
    assert cloud_run_monthly_usd(1, 0.5, 2, "request", 18_000_000) > cloud_run_monthly_usd(1, 0.5, 2, "instance")
    print(f"   orchestrator fleet (3 × 2 vCPU / 2 GiB, instance-based): ${cloud_run_monthly_usd(2, 2, 3, 'instance'):,.0f}/month vs model bill ${p['cost']['per_month_routed_cached']:,.0f}")
check("c: Cloud Run cost", _c)

#### (d) Choose the cap

Pick an in-flight cap for the 3 M TPM pool with 120 users so that p95 ≤ 8 s and no turn fails; keep shedding under 30 %. The check runs a simulation with your value.

In [ ]:
MY_CAP = 40

In [ ]:
async def _d():
    if MY_CAP is None:
        todo()
    CLOCK.reset(0.02)
    r = await simulate(make_setup(pool_tpm=3_000_000, max_inflight=MY_CAP), users=120, duration_s=60, think_s=5)
    s = r.summary()
    print(f"   cap {MY_CAP}: p95 {s['p95_s']:.1f} s, shed {s['shed_rate']:.0%}, failed {s['failed']}, 429s {s['rate_limited_calls']}")
    assert s["p95_s"] <= 8 and s["failed"] == 0 and s["shed_rate"] < 0.30
await acheck("d: choose the cap", _d)

## Takeaways

- A load test gives you the turn duration; Little's law gives you in-flight turns; headroom and per-instance concurrency give you instances.
- The naive system does not fail — it slows down for everyone. Admission control converts that into bounded latency plus honest `Retry-After`s.
- Instance-based billing on the orchestrator (background work after the response); request-based on the gateway.
- Say what breaks first: the token budget, then the slowest downstream system; Cloud Run is not on the list.

## Verify before relying on it

Cloud Run limits (concurrency 1,000; 800 req/s per instance; 60-minute request ceiling; Direct VPC egress instance quota), pricing, and whether custom autoscaling targets and Cloud Run *instances* have left Preview.